# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Mã chính nằm trong `lab/vlm.py` (nhãn JSON, suy luận, mask loss, setup LoRA, vòng train). Notebook này gọi nó theo thứ tự GUIDE; train/suy luận chạy nền (`nohup`) để Colab không ngắt cell dài.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!git clone -q https://github.com/BenKei-19/K4-Track-4-Day10-VisionLanguageModel /content/lab
%cd /content/lab
!pip -q install -r requirements-colab.txt
# torchao 0.10 cài sẵn trên Colab không tương thích peft 0.21.2 (cần > 0.16); lab không dùng torchao
!pip uninstall -y -q torchao

In [ ]:
# Kaggle: Colab Secrets (token API kiểu mới, kaggle CLI >= 1.8). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle
!python -m pytest -q

In [ ]:
import json, collections
from PIL import Image, ImageDraw
from scripts.labcommon import read_jsonl
S = {s: read_jsonl(f'data/splits/{s}.jsonl') for s in ('train', 'val', 'test')}
for s, rows in S.items():
    c = collections.Counter(o['label'] for r in rows for o in r['objects'])
    print(s, len(rows), 'ảnh', dict(c))
sess = {s: {r['session'] for r in rows} for s, rows in S.items()}
assert not (sess['train'] & sess['val'] or sess['train'] & sess['test'] or sess['val'] & sess['test'])
print('không session nào nằm ở hai split')
COL = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}
tiles = []
for r in [r for r in S['train'] if r['objects']][::200][:6]:
    im = Image.open('data/' + r['image_path']).convert('RGB'); d = ImageDraw.Draw(im)
    for o in r['objects']:
        xc, yc, w, h = o['yolo']
        d.rectangle([(xc - w/2) * 640, (yc - h/2) * 360, (xc + w/2) * 640, (yc + h/2) * 360], outline=COL[o['label']], width=3)
    tiles.append(im)
grid = Image.new('RGB', (1280, 1080))
for i, t in enumerate(tiles): grid.paste(t, ((i % 2) * 640, (i // 2) * 360))
grid

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# yolo_to_box(), objects_to_answer(), prompt() nằm trong lab/vlm.py.
# `check` kiểm tra: round-trip nhãn → pixel khớp box gốc, 220 token ảnh, labels != -100 giải mã đúng JSON + <|im_end|>
!python lab/vlm.py check

## 3. E0: zero-shot

In [ ]:
# load_model(): NF4 4-bit, skip 'model.visual' và 'lm_head' (xem chú thích trong lab/vlm.py về re.match)
!python lab/vlm.py predict --name e0   # → work/e0.jsonl và tự gọi score.py

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# encode_example(): prompt phải là tiền tố của chuỗi đầy đủ, labels[:n_prompt] = -100
# collate(): pad input_ids/attention_mask/mm_token_type_ids/labels (labels đệm -100), cat pixel_values/image_grid_thw
import inspect, lab.vlm as V
print(inspect.getsource(V.encode_example)); print(inspect.getsource(V.collate))

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# apply_setup(): regex LLM (252 module) và LLM+ViT (348 module); E2/E3 mở projector ở float32
print(V.LLM_RX); print(V.VIT_RX); print(V.PROJECTOR)
# chạy thử 3 bước với setup nặng nhất để bắt lỗi sớm
!python lab/vlm.py train --name smoke --targets llm+vit --projector --steps 3 --out-dir /content/smoke
!python lab/vlm.py predict --name smoke --adapter /content/smoke/smoke --split val --limit 8

## 6. Vòng huấn luyện

In [ ]:
# AdamW lr 2e-4, warmup 5% + cosine, fp16 + GradScaler, batch 2 × accum 4, 300 bước, lưu adapter + projector.pt + run.json
!nohup sh -c "python lab/vlm.py train --name e1 --targets llm && \n  python lab/vlm.py train --name e2 --targets llm --projector && \n  python lab/vlm.py train --name e3 --targets llm+vit --projector" > /content/train.log 2>&1 &

In [ ]:
!tail -c 2000 /content/train.log

## 7. Suy luận và chấm E1–E3

In [ ]:
# predict nạp model gốc → adapter → projector.pt, rồi gọi score.py với --meta lấy từ run.json
for n in ('e1', 'e2', 'e3'):
    !python lab/vlm.py predict --name {n} --adapter work/adapters/{n}

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: data_size (10% vs 100% ảnh train) trên setup tốt nhất của E1–E3. Giá trị 1.0 chính là setup gốc.
BEST = 'e1'  # đổi theo kết quả bảng E1–E3
FLAGS = {'e1': '--targets llm', 'e2': '--targets llm --projector', 'e3': '--targets llm+vit --projector'}[BEST]
!python lab/vlm.py train --name e4_data_size_0.1 {FLAGS} --fraction 0.1
!python lab/vlm.py predict --name e4_data_size_0.1 --adapter work/adapters/e4_data_size_0.1
# data_size=1.0 là đúng lần chạy BEST: dùng lại bằng chứng của nó, chỉ đổi tên
ev = json.load(open(f'report/evidence/{BEST}.json', encoding='utf-8')); ev['name'] = 'E4_DATA_SIZE_1.0'
json.dump(ev, open('report/evidence/e4_data_size_1.0.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
# E5 (GRPO) không làm.